# Five-class vehicle classifier

This notebook trains a transfer-learning image classifier for airplanes, buses, cars, motorcycles, and ships. Run the cells from top to bottom after selecting the project `.venv` kernel.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["airplanes", "buses", "cars", "motorcycles", "ship"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 16
TRAIN_DIR = Path("Datasets/train/images_datasets")
TEST_DIR = Path("Datasets/test")
MODEL_PATH = Path("best_vehicle_model.keras")

print("TensorFlow:", tf.__version__)
print("Classes:", CLASS_NAMES)

## Validate the dataset

The checks below stop early if a class folder is missing or empty.

In [ ]:
def image_count(folder):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    return sum(
        path.is_file() and path.suffix.lower() in extensions
        for path in folder.iterdir()
    )


for root in (TRAIN_DIR, TEST_DIR):
    if not root.exists():
        raise FileNotFoundError(f"Dataset directory not found: {root}")

    unexpected = sorted(
        folder.name
        for folder in root.iterdir()
        if folder.is_dir() and folder.name not in CLASS_NAMES
    )
    if unexpected:
        print(f"Warning: unused folders in {root}: {unexpected}")

    for class_name in CLASS_NAMES:
        class_dir = root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")
        count = image_count(class_dir)
        if count == 0:
            raise ValueError(f"No images found in: {class_dir}")
        print(f"{class_dir}: {count} images")

## Load training, validation, and test data

Augmentation is applied only to training images. Validation and test images remain unchanged. Pixel preprocessing is built into the model, so prediction uses exactly the same transformation.

In [ ]:
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)

validation_datagen = keras.preprocessing.image.ImageDataGenerator(
    validation_split=0.20
)

test_datagen = keras.preprocessing.image.ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

validation_generator = validation_datagen.flow_from_directory(
    TRAIN_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED,
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

assert list(train_generator.class_indices) == CLASS_NAMES
assert list(test_generator.class_indices) == CLASS_NAMES
print("Class indices:", train_generator.class_indices)

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(12, 8))
for index in range(min(12, len(images))):
    plt.subplot(3, 4, index + 1)
    plt.imshow(images[index].astype("uint8"))
    plt.title(CLASS_NAMES[int(np.argmax(labels[index]))])
    plt.axis("off")
plt.tight_layout()
plt.show()

train_generator.reset()

## Build a transfer-learning model

MobileNetV2 starts with ImageNet visual features. Global average pooling keeps the model small and reduces overfitting compared with a large flattened dense layer.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,), name="image")
x = layers.Rescaling(1.0 / 127.5, offset=-1, name="mobilenet_preprocessing")(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="vehicle_classifier")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## Train and save the best model

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=25,
    callbacks=callbacks,
    verbose=2,
)

model = keras.models.load_model(MODEL_PATH)

with open("class_names.json", "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=2)

print(f"Best model saved to: {MODEL_PATH}")

In [ ]:
epochs_run = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_run, history.history["accuracy"], label="Training")
plt.plot(epochs_run, history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_run, history.history["loss"], label="Training")
plt.plot(epochs_run, history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Evaluate the held-out test set

In [ ]:
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
probabilities = model.predict(test_generator, verbose=0)
predicted_classes = np.argmax(probabilities, axis=1)
true_classes = test_generator.classes

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}\n")
print(
    classification_report(
        true_classes,
        predicted_classes,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_predictions(
    true_classes,
    predicted_classes,
    display_labels=CLASS_NAMES,
    cmap="Blues",
    xticks_rotation=45,
)
plt.title("Test confusion matrix")
plt.tight_layout()
plt.show()

## Predict one image

In [ ]:
def test_single_image(image_path, model_path=MODEL_PATH):
    image_path = Path(image_path)
    model_path = Path(model_path)

    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}. Train the model first.")

    prediction_model = keras.models.load_model(model_path)
    if prediction_model.output_shape[-1] != len(CLASS_NAMES):
        raise ValueError("The saved model does not match the configured class list.")

    image = keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    image_array = keras.utils.img_to_array(image)
    image_batch = np.expand_dims(image_array, axis=0)

    probabilities = prediction_model.predict(image_batch, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(probabilities[predicted_index])

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title(f"Predicted: {predicted_class}\nConfidence: {confidence:.2%}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    colors = ["crimson" if i == predicted_index else "steelblue" for i in range(NUM_CLASSES)]
    bars = plt.bar(CLASS_NAMES, probabilities * 100, color=colors)
    plt.title("Class probabilities")
    plt.ylabel("Probability (%)")
    plt.ylim(0, 105)
    plt.xticks(rotation=35, ha="right")
    for bar, probability in zip(bars, probabilities):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{probability:.1%}",
            ha="center",
        )
    plt.tight_layout()
    plt.show()

    print(f"Image: {image_path.name}")
    print(f"Predicted class: {predicted_class}")
    print(f"Confidence: {confidence:.2%}")
    for label, probability in zip(CLASS_NAMES, probabilities):
        print(f"{label:<12}: {probability:.2%}")

    return predicted_class, confidence, probabilities

In [ ]:
test_single_image("Datasets/test/ship/ship_005.jpg")